# Engineer arrival-model features

Create the deterministic Appendix B features for Models 2A, 2B, and 2C. The output is their union, while `feature_engineering.py` provides separate prediction-time-safe candidate allowlists for each model. No learned preprocessing is performed here.

In [ ]:
YEAR = 2019

AIRPORT = "JFK"

## Configure the merged input and feature output

The notebook works from either the project root or the `notebooks` directory. `AIRPORT` is normalized to uppercase, and the input must already contain arrivals at that airport.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

if (Path.cwd() / "data").is_dir() and (Path.cwd() / "notebooks").is_dir():
    PROJECT_ROOT = Path.cwd()
elif Path.cwd().name == "notebooks" and (Path.cwd().parent / "data").is_dir():
    PROJECT_ROOT = Path.cwd().parent
else:
    raise FileNotFoundError(
        "Start this notebook from the capstone project root or its notebooks directory"
    )

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

from feature_engineering import (
    ALL_ARRIVAL_ENGINEERED_FEATURES,
    MODEL_TARGETS,
    POST_PUSHBACK_ENGINEERED_FEATURES,
    POST_TAKEOFF_ENGINEERED_FEATURES,
    PRE_ENGINEERED_FEATURES,
    add_all_arrival_features,
    model_feature_candidates,
    validate_engineered_features,
)

AIRPORT = str(AIRPORT).strip().upper()
YEAR = int(YEAR)
INPUT_FILE = PROJECT_ROOT / f"data/merged/{AIRPORT}_{YEAR}_arrivals.csv"
OUTPUT_FILE = PROJECT_ROOT / f"data/features/{AIRPORT}_{YEAR}_arrivals.csv"

print(pd.Series({"input": str(INPUT_FILE), "output": str(OUTPUT_FILE)}))

## Load and validate the arrival population

The notebook validates rather than silently filtering the merged dataset. The arrival target must be complete and binary before feature generation begins.

In [ ]:
if not INPUT_FILE.is_file():
    raise FileNotFoundError(f"Merged arrival file does not exist: {INPUT_FILE}")

source = pd.read_csv(INPUT_FILE, low_memory=False)
required_scope_columns = {"Year", "Dest", MODEL_TARGETS["2A"]}
missing_scope_columns = required_scope_columns - set(source.columns)
if missing_scope_columns:
    raise KeyError(f"Merged arrival data is missing: {sorted(missing_scope_columns)}")

destination = source["Dest"].astype("string").str.strip().str.upper()
source_year = pd.to_numeric(source["Year"], errors="coerce")
if not destination.eq(AIRPORT).all():
    raise ValueError(f"Arrival input contains destinations other than {AIRPORT}")
if not source_year.eq(YEAR).all():
    raise ValueError(f"Arrival input contains years other than {YEAR}")

target = pd.to_numeric(source[MODEL_TARGETS["2A"]], errors="coerce")
if target.isna().any() or not target.isin([0, 1]).all():
    raise ValueError("ArrDel15 must be complete and binary")

print(pd.Series({"rows": len(source), "columns": len(source.columns), "delayed": int(target.sum())}))

## Add the Model 2A, 2B, and 2C features

The output contains the union of all three horizons. Missing values propagate from their inputs, and later model code must select its predictors using the corresponding allowlist.

In [ ]:
features = add_all_arrival_features(source)
if len(features) != len(source):
    raise ValueError("Feature engineering changed the arrival row count")
if not features[source.columns].equals(source):
    raise ValueError("Feature engineering changed one or more source columns")

feature_validation = validate_engineered_features(
    features, ALL_ARRIVAL_ENGINEERED_FEATURES
)
candidate_columns = {
    model: model_feature_candidates(model, features.columns)
    for model in ("2A", "2B", "2C")
}
if not set(candidate_columns["2A"]) <= set(candidate_columns["2B"]):
    raise ValueError("Model 2A candidates are not a subset of Model 2B candidates")
if not set(candidate_columns["2B"]) <= set(candidate_columns["2C"]):
    raise ValueError("Model 2B candidates are not a subset of Model 2C candidates")
if any(MODEL_TARGETS[model] in columns for model, columns in candidate_columns.items()):
    raise ValueError("Arrival target leaked into a feature allowlist")

print(pd.Series({
    "Pre engineered features": len(PRE_ENGINEERED_FEATURES),
    "post-pushback engineered features": len(POST_PUSHBACK_ENGINEERED_FEATURES),
    "post-takeoff engineered features": len(POST_TAKEOFF_ENGINEERED_FEATURES),
    "Model 2A candidate columns": len(candidate_columns["2A"]),
    "Model 2B candidate columns": len(candidate_columns["2B"]),
    "Model 2C candidate columns": len(candidate_columns["2C"]),
    "rows with missing engineered values": int(
        features[ALL_ARRIVAL_ENGINEERED_FEATURES].isna().any(axis=1).sum()
    ),
}))

In [ ]:
feature_validation

## Save the arrival feature dataset

The CSV retains all audit fields, the arrival target, and the later operating fields needed by Models 2B and 2C. Those later fields remain prohibited from earlier prediction horizons.

In [ ]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
features.to_csv(OUTPUT_FILE, index=False)

summary = pd.Series({
    "airport": AIRPORT,
    "year": YEAR,
    "rows": len(features),
    "columns": len(features.columns),
    "target": MODEL_TARGETS["2A"],
    "output": str(OUTPUT_FILE),
}, name="arrival feature summary")
print(f"Saved {len(features):,} rows to {OUTPUT_FILE}")
summary